# Analysis - what it writes for one odor, beside the molecules that carry it

Issue #10. Forty molecules for `fruity` and forty for `musk`, drawn on this machine from the committed generator,
compared with the training molecules a source actually named with that token.

The numbers come from `python -m scripts.analysis.generated_vs_real`; nothing here trains.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = json.loads((ROOT / "output/analysis/generated_vs_real.json").read_text())
requests = data["requests"]
pd.DataFrame(
    [
        {
            "odor": request,
            "real molecules": found["real_molecules"],
            "written": found["generated"],
            "similarity to real": round(found["similarity_to_real"]["median"], 2),
            "similarity to any training": round(found["similarity_to_any_training"]["median"], 2),
            "generated weight": round(found["weight"]["generated_median"]),
            "real weight": round(found["weight"]["real_median"]),
        }
        for request, found in requests.items()
    ]
).set_index("odor")

## 1. Is it close to the molecules that carry the odor, or just to chemistry?

Tanimoto over Morgan fingerprints, for each written molecule, to the nearest training molecule carrying the token
and to the nearest training molecule of any kind.

In [ ]:
fig, axes = plt.subplots(1, len(requests), figsize=(10, 3.2), sharey=True)
for ax, (request, found) in zip(axes, requests.items(), strict=True):
    ax.hist(found["similarity_to_real_values"], bins=15, alpha=0.75, label="nearest that smell of it", color="#1F3864")
    ax.hist(found["similarity_to_any_values"], bins=15, alpha=0.55, label="nearest of any kind", color="#8FAADC")
    ax.set_title(request)
    ax.set_xlabel("Tanimoto to the nearest training molecule")
axes[0].set_ylabel("written molecules")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 2. What it wrote, beside what exists

Five of each, for the two odors. The SMILES are the whole comparison: no smell was ever measured.

In [ ]:
for request, found in requests.items():
    print(request.upper(), "- written")
    for smiles in found["examples"][:5]:
        print("   ", smiles)
    print(request.upper(), "- real")
    for smiles in found["real_examples"][:5]:
        print("   ", smiles)
    print()

## Reading.

The two odors answer differently, and neither answer is the one a reader hopes for.

For `fruity`, 2,236 training molecules carry the token, and a written molecule sits 0.67 from the nearest of them on
average. It sits 0.68 from the nearest training molecule of any kind, so the odor adds nothing: the generator is
producing ordinary fine-tuning chemistry, and a fruity molecule is simply the sort of thing that is nearby anyway.

For `musk`, only 180 molecules carry the token, and the failure is the other way round: a written molecule is 0.48
from the nearest musk molecule but 0.61 from training chemistry in general. The molecules it writes for musk are
further from real musk molecules than from the corpus, which is what a weak conditioner looks like. The weights are
right in both cases, 196 against 182 Da for fruity and 236 against 237 for musk, so the model has learned the size
of the right answer and not its neighbourhood.

## Verdict

Structure similarity cannot confirm that the generator writes what was asked for. It writes plausible molecules,
of the right weight, and for a rare odor they are not even close to the real examples. The only check that could
settle it is smelling them, and that is why the report treats this as the project's open question rather than a
result. The `novel` column in the evaluation says the same thing from the other side: 70 to 86 percent of these
structures are new, which here means new and unverified.
